In [1]:
import mysql.connector

In [3]:
from tabulate import tabulate

In [4]:
conn = mysql.connector.connect(
    host="localhost",       # MySQL server host
    database="app",         # database name
    user="warlord",            # MySQL username
    password="Warlord@200206"  # MySQL password
)

cursor = conn.cursor()


In [7]:
from tabulate import tabulate
def table_view(cursor):
    rows = cursor.fetchall()
    col_names = [desc[0] for desc in cursor.description]

    print(tabulate(rows, headers=col_names, tablefmt="grid"))
    

In [8]:
cursor.execute("SELECT * FROM student;")
table_view(cursor)

+-----------+-------------+--------+---------+-------------+-------------+
| stud_id   | stud_name   | year   | email   | course_id   | hash_pass   |
+===========+=============+========+=========+=============+=============+
+-----------+-------------+--------+---------+-------------+-------------+


In [90]:
cursor.execute("DROP TABLE staff;")

In [77]:
cursor.execute("SHOW TABLES;")
print(cursor)

CMySQLCursor: SHOW TABLES;


In [66]:
cursor.execute("CREATE TABLE course(`course_id` varchar(6) PRIMARY KEY, `course_name` varchar(50));")

In [84]:
cursor.execute("CREATE TABLE student (`stud_id` varchar(8) PRIMARY KEY, `stud_name` TEXT, `year` INT, `email` varchar(30), `course_id` varchar(6), FOREIGN KEY(course_id)  REFERENCES course(course_id));")

In [91]:
cursor.execute("CREATE TABLE module (`mod_id` varchar(8) PRIMARY KEY, `mod_name` TEXT, `course_id` varchar(6), `year` INT, `welfare_staff_id` varchar(8), `module_lead_id` varchar(8) , FOREIGN KEY (welfare_staff_id) REFERENCES welfare_staff(staff_id) ,FOREIGN KEY (module_lead_id) REFERENCES module_staff(staff_id) ,FOREIGN KEY (course_id) REFERENCES course(course_id));")

In [85]:
cursor.execute("CREATE TABLE module_staff (`staff_id` varchar(8) PRIMARY KEY, `staff_name` TEXT, `email` varchar(30));")

In [11]:
cursor.execute("SELECT * FROM users;")
table_view(cursor)


+-----------+---------------+-----------------+
| user_id   | role          | email           |
+===========+===============+=================+
| S0000001  | welfare_staff | welfare@uni.edu |
+-----------+---------------+-----------------+
| S0000002  | module_staff  | module@uni.edu  |
+-----------+---------------+-----------------+
| S0000003  | welfare_staff | wrong@uni.edu   |
+-----------+---------------+-----------------+


In [98]:
cursor.execute("CREATE TABLE attendance (`week_no` INT, `mod_id` varchar(8), `stud_id` varchar(8) , `att_per` DECIMAL(4,2) , UNIQUE KEY unique_stu_att_per_week(week_no, mod_id, stud_id), FOREIGN KEY (stud_id) REFERENCES student(stud_id) , FOREIGN KEY (mod_id) REFERENCES module(mod_id));")

In [101]:
cursor.execute("CREATE TABLE surveys (`sur_id` INT PRIMARY KEY, `week_no` INT, `stud_id` varchar(8), `mod_id` varchar(8), `stress_levels` INT CHECK(`stress_levels` > 0 AND `stress_levels` <=5), `hours_slept` FLOAT CHECK(`hours_slept` < 24), `comments` varchar(200) DEFAULT 'NO COMMENTS', UNIQUE KEY unique_stu_per_week (week_no, stud_id, mod_id), FOREIGN KEY (stud_id) REFERENCES student(stud_id), FOREIGN KEY (mod_id) REFERENCES module(mod_id));")

In [103]:
cursor.execute("CREATE TABLE deadlines (`dead_id` INT PRIMARY KEY, `mod_id` varchar(8) , `week_no` INT, `ass_name` varchar(100) DEFAULT 'Assessment', `due_date` DATE, FOREIGN KEY (mod_id) REFERENCES module(mod_id));")

In [86]:
cursor.execute("CREATE TABLE welfare_staff (`staff_id` varchar(8) PRIMARY KEY, `staff_name` TEXT, `email` varchar(30));")

In [53]:
conn.commit()

In [19]:
cursor.execute("CREATE table student(`Student_id` INT PRIMARY KEY, `Student_name` TEXT);")

In [24]:
cursor.execute("CREATE table module(`module_id` INT PRIMARY KEY, `module_name` TEXT, `year` INT);")

In [7]:
cursor.execute("CREATE table users(`user_id` varchar(8), `role` ENUM ('module_staff','welfare_staff') NOT NULL, `email` varchar(30) );")

In [ ]:
query = "SELECT * FROM student WHERE stud_id = %s;"
cursor.execute(query, (stud_id,))
existing = cursor.fetchone()

In [32]:
import bcrypt

class Student:
    def __init__(self, cursor, conn):
        self.cursor = cursor
        self.conn = conn

    def register(self, stud_id, stud_name, password):
        try:
            # Hash password
            hashed_pass = bcrypt.hashpw(password.encode(), bcrypt.gensalt())

            query = "SELECT * FROM student WHERE stud_id = %s;"
            self.cursor.execute(query, (stud_id,))
            existing = self.cursor.fetchone()

            if existing is None:
                query = "INSERT INTO student (stud_id, stud_name, password) VALUES (%s, %s, %s);"
                self.cursor.execute(query, (stud_id, stud_name, hashed_pass))
                self.conn.commit()
                print("Student registered successfully!")
            else:
                print("Student already registered!")
        except Exception as e:
            print("ERROR:", e)



In [33]:
class staff:
    def __init__(self,cursor,conn):
        self.cursor = cursor
        self.conn = conn
    def register(self, staff_id, staff_name, password):
        try:
            hashed_pass = bcrypt.hashpw(password.encode(), bcrypt.gensalt())

            query = "SELECT * FROM staff WHERE staff_id = %s;"
            self.cursor.execute(query, (staff_id,))
            existing = self.cursor.fetchone()

            if existing is None:
                query = "INSERT INTO staff (staff_id, staff_name, password) VALUES (%s, %s, %s);"
                self.cursor.execute(query, (staff_id, staff_name, hashed_pass))
                self.conn.commit()
                print("Staff registered successfully!")
            else:
                print("Staff already registered!")
        except Exception as e:
            print("ERROR:", e)


In [34]:
class read_stud:
    def __init__(self,cursor,conn):
        self.cursor= cursor
        self.conn = conn
    def read_table(self):
        self.cursor.execute("SELECT * FROM student;")
        table_view(self.cursor)
    def read_col_name(self, col_name, value):
        query = f"SELECT * FROM student WHERE `{col_name}` = %s;"
        self.cursor.execute(query, (value,))
        table_view(self.cursor)


In [ ]:
class Update_row_stud:
    def __init__(self,cursor,conn):
        self.cursor= cursor
        self.conn = conn
    def update_row_by_id(self,user, new_var):
        query = f"""
            UPDATE student
            SET `{user.col_name}` = %s
            WHERE `stud_id` = %s;
        """
        self.cursor.execute(query, (new_var, user.stud_id))
        self.conn.commit()
    def del_row_by_id(self,user):
        query = f"""
            DELETE FROM student
            WHERE `stud_id` = %s;"""
        self.cursor.execute(query,(user.stud_id,))
        self.conn.commit()
    def update_col(self,old_colname, new_colname):
        query=f"""
            ALTER TABLE student
            RENAME COLUMN `{old_colname}` TO `{new_colname}`;
            """
        self.cursor.execute(query)
        self.conn.commit()
    def del_col(self, colname):
        query = f"""
            ALTER TABLE student
            DROP COLUMN `{colname}`;
            """
        self.cursor.execute(query)
        self.conn.commit()
    

In [ ]:
class update_col_stud:
    def __init__(self,cursor,conn): 
        self.cursor= cursor
        self.conn = conn
    


In [37]:
class read_staff:
    def __init__(self,cursor,conn):
        self.cursor= cursor
        self.conn = conn
    def read_table(self):
        self.cursor.execute("SELECT * FROM staff;")
        table_view(self.cursor)
    def read_col_name(self,col_name):
        query = f"""
            SELECT * FROM staff 
            WHERE  `{col_name}` = %s;
            """
        self.cursor.execute(query,(col_name,))
        table_view(self.cursor)
    

In [38]:
class update_row_staff:
    def __init__(self,cursor,conn):
        self.cursor= cursor
        self.conn = conn
    def update_row(self,col_name, staff_id, new_var):
        query = f"""
            UPDATE staff
            SET `{col_name}` = %s
            WHERE `staff_id` = %s;
        """
        self.cursor.execute(query, (new_var, staff_id))
        self.conn.commit()
    def del_row(self, staff_id):
        query = "DELETE FROM staff WHERE staff_id = %s;"
        self.cursor.execute(query, (staff_id,))
        self.conn.commit()

    

In [39]:
class update_col_staff:
    def __init__(self,cursor,conn):
        self.cursor= cursor
        self.conn = conn
    def update_col(self,old_colname, new_colname):
        query=f"""
            ALTER TABLE staff
            RENAME COLUMN `{old_colname}` TO `{new_colname}`;
            """
        self.cursor.execute(query)
        self.conn.commit()
    def del_col(self, colname):
        query = f"""
            ALTER TABLE staff
            DROP COLUMN `{colname}`;
            """
        self.cursor.execute(query)
        self.conn.commit()


cursor.execute("CREATE TABLE student(`stud_id` varchar(8) PRIMARY KEY, `stud__name` varchar(30), `Age` INT, `Weekly attendance` FLOAT CHECK(`Weekly attendance` BETWEEN 0 AND 100), `Submission deadlines` INT, `Hours slept` FLOAT,`Stress levels` INT CHECK(`Stress levels` BETWEEN 0 AND 10), `Module satisfaction` INT CHECK(`Module satisfaction` BETWEEN 0 AND 10))")
conn.commit()

cursor.execute("CREATE TABLE staff(`staff_id` varchar(8) PRIMARY KEY, `staff__name` varchar(30), `Age` INT, `Weekly attendance` FLOAT CHECK(`Weekly attendance` BETWEEN 0 AND 100), `Academic deadlines` INT, `Hours slept` FLOAT,`Stress levels` INT CHECK(`Stress levels` BETWEEN 0 AND 10), `Module readiness` INT CHECK(`Module readiness` BETWEEN 0 AND 10))")
conn.commit()

In [40]:
cursor.execute("SELECT * FROM staff")
table_view(cursor)

+------------+---------------+-------+---------------------+----------------------+---------------+-----------------+--------------------+
| staff_id   | staff__name   | Age   | Weekly attendance   | Academic deadlines   | Hours slept   | Stress levels   | Module readiness   |
+============+===============+=======+=====================+======================+===============+=================+====================+
+------------+---------------+-------+---------------------+----------------------+---------------+-----------------+--------------------+


In [45]:
id = input("Enter your student id: ")
name = input("Enter your name: ")
password = input("Enter your password: ")
pass_bytes = password.encode('utf-8')
hash_pass = bcrypt.hashpw(pass_bytes,bcrypt.gensalt())
print(f"Your id: {id}, your name: {name}, your pass: {password}, and your hashed password: {hash_pass}")


Your id: u5989662, your name: shiv, your pass: Warlord@2--2, and your hashed password: b'$2b$12$xy4SgbPG1WlQL/OlYNmDRORWfHQGvsJvFPv39VpEVe3VTVvRhmJZy'


In [46]:
stu = Student(cursor,conn)
stu.register(id,name,password)


Student registered successfully!


In [26]:
stu1 = Update_row_stud(cursor,conn)
password = bcrypt.hashpw(password.encode(), bcrypt.gensalt())
stu1.update_row("password", "u5896985", password)

In [41]:
cursor.execute("SELECT * FROM student ORDER BY stud_name;")
table_view(cursor)

+-----------+-------------+-------+---------------------+------------------------+---------------+-----------------+-----------------------+--------------------------------------------------------------+
| stud_id   | stud_name   | Age   | Weekly attendance   | Submission deadlines   | Hours slept   | Stress levels   | Module satisfaction   | password                                                     |
+===========+=============+=======+=====================+========================+===============+=================+=======================+==============================================================+
| u5696991  | helloo      |       |                     |                        |               |                 |                       | $2b$12$Sm8tKAGJ5yVQG9zO40QXAedSiGrjwDPzup2pifaem/rzZjTkrvvxu |
+-----------+-------------+-------+---------------------+------------------------+---------------+-----------------+-----------------------+--------------------------------------------